In [3]:
!nvcc --version
import torch
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_available())
print(torch.version.cuda)

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0
Tesla T4
True
12.8


In [4]:
%%writefile matmul.cu

#include <stdio.h>
#include <cuda_runtime.h>

// ── Naive matmul kernel ────────────────────────────────────────────────────
// Each thread computes one element of C = A @ B
// A: (M, K)  B: (K, N)  C: (M, N)
__global__ void matmul_naive(float* A, float* B, float* C,
                              int M, int K, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < M && col < N) {
        float acc = 0.0f;
        for (int k = 0; k < K; k++) {
            acc += A[row * K + k] * B[k * N + col];
        }
        C[row * N + col] = acc;
    }
}

// ── Tiled matmul kernel ────────────────────────────────────────────────────
// Threads cooperate to load tiles into shared memory before computing
#define TILE 16

__global__ void matmul_tiled(float* A, float* B, float* C,
                              int M, int K, int N) {
    __shared__ float As[TILE][TILE];
    __shared__ float Bs[TILE][TILE];

    int row = blockIdx.y * TILE + threadIdx.y;
    int col = blockIdx.x * TILE + threadIdx.x;
    float acc = 0.0f;

    for (int t = 0; t < (K + TILE - 1) / TILE; t++) {
        // Load tile of A into shared memory
        if (row < M && t * TILE + threadIdx.x < K)
            As[threadIdx.y][threadIdx.x] = A[row * K + t * TILE + threadIdx.x];
        else
            As[threadIdx.y][threadIdx.x] = 0.0f;

        // Load tile of B into shared memory
        if (col < N && t * TILE + threadIdx.y < K)
            Bs[threadIdx.y][threadIdx.x] = B[(t * TILE + threadIdx.y) * N + col];
        else
            Bs[threadIdx.y][threadIdx.x] = 0.0f;

        __syncthreads();  // Wait for all threads to finish loading

        // Compute partial dot product from tile
        for (int k = 0; k < TILE; k++)
            acc += As[threadIdx.y][k] * Bs[k][threadIdx.x];

        __syncthreads();  // Wait before loading next tile
    }

    if (row < M && col < N)
        C[row * N + col] = acc;
}

// ── Main: benchmark both kernels ──────────────────────────────────────────
int main() {
    // Matrix dims matching your BC policy linear layers
    // Largest layer: (1000, 19) @ (19, 256) and (1000, 256) @ (256, 256)
    int M = 1000, K = 256, N = 256;

    size_t bytes_A = M * K * sizeof(float);
    size_t bytes_B = K * N * sizeof(float);
    size_t bytes_C = M * N * sizeof(float);

    // Allocate host memory
    float *h_A = (float*)malloc(bytes_A);
    float *h_B = (float*)malloc(bytes_B);
    float *h_C = (float*)malloc(bytes_C);

    // Initialize with random values
    for (int i = 0; i < M * K; i++) h_A[i] = (float)rand() / RAND_MAX;
    for (int i = 0; i < K * N; i++) h_B[i] = (float)rand() / RAND_MAX;

    // Allocate device memory
    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, bytes_A);
    cudaMalloc(&d_B, bytes_B);
    cudaMalloc(&d_C, bytes_C);

    cudaMemcpy(d_A, h_A, bytes_A, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, bytes_B, cudaMemcpyHostToDevice);

    // Grid and block dimensions
    dim3 block(TILE, TILE);
    dim3 grid((N + TILE - 1) / TILE, (M + TILE - 1) / TILE);

    // CUDA events for timing
    cudaEvent_t start, stop;
    cudaEventCreate(&start);
    cudaEventCreate(&stop);
    float ms = 0.0f;

    // ── Warmup ────────────────────────────────────────────────────────────
    for (int i = 0; i < 5; i++) {
        matmul_naive<<<grid, block>>>(d_A, d_B, d_C, M, K, N);
        matmul_tiled<<<grid, block>>>(d_A, d_B, d_C, M, K, N);
    }
    cudaDeviceSynchronize();

    // ── Benchmark naive ───────────────────────────────────────────────────
    cudaEventRecord(start);
    for (int i = 0; i < 100; i++)
        matmul_naive<<<grid, block>>>(d_A, d_B, d_C, M, K, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Naive matmul:  %.3f ms avg  (%.2f GFLOP/s)\n",
           ms / 100, 2.0f * M * K * N / (ms / 100 * 1e6));

    // ── Benchmark tiled ───────────────────────────────────────────────────
    cudaEventRecord(start);
    for (int i = 0; i < 100; i++)
        matmul_tiled<<<grid, block>>>(d_A, d_B, d_C, M, K, N);
    cudaEventRecord(stop);
    cudaEventSynchronize(stop);
    cudaEventElapsedTime(&ms, start, stop);
    printf("Tiled matmul:  %.3f ms avg  (%.2f GFLOP/s)\n",
           ms / 100, 2.0f * M * K * N / (ms / 100 * 1e6));

    // Cleanup
    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);
    cudaEventDestroy(start); cudaEventDestroy(stop);

    return 0;
}

Writing matmul.cu


In [5]:
!nvcc -O2 -o matmul matmul.cu
!./matmul

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Naive matmul:  0.413 ms avg  (317.34 GFLOP/s)
Tiled matmul:  0.287 ms avg  (457.16 GFLOP/s)


In [7]:
%%writefile benchmark_cublas.py
import torch
import time

device = torch.device('cuda')

# Same matrix dims as the CUDA kernel
M, K, N = 1000, 256, 256
A = torch.randn(M, K, device=device)
B = torch.randn(K, N, device=device)

# Warmup
for _ in range(10):
    C = torch.mm(A, B)
torch.cuda.synchronize()

# Benchmark
start = torch.cuda.Event(enable_timing=True)
end   = torch.cuda.Event(enable_timing=True)

start.record()
for _ in range(100):
    C = torch.mm(A, B)
end.record()
torch.cuda.synchronize()

ms = start.elapsed_time(end) / 100
gflops = 2 * M * K * N / (ms * 1e6)
print(f"cuBLAS (torch.mm): {ms:.3f} ms avg  ({gflops:.2f} GFLOP/s)")

Overwriting benchmark_cublas.py


In [8]:
!python benchmark_cublas.py

cuBLAS (torch.mm): 0.057 ms avg  (2310.33 GFLOP/s)


"A naive CUDA matmul kernel achieved 317 GFLOP/s on the T4, consistent with poor memory reuse from global memory accesses. Introducing 16×16 shared memory tiling improved throughput to 457 GFLOP/s — a 1.44x speedup — by reducing redundant DRAM reads through tile-level data reuse. However, both implementations remain 5-7x behind cuBLAS (2310 GFLOP/s), which employs architecture-specific optimizations including larger tile sizes, vectorized loads, and warp-level primitives. This gap motivates the Triton kernel in Week 7, which provides higher-level control over memory hierarchy without raw CUDA complexity."

In [9]:
import triton
print(triton.__version__)

3.6.0


In [12]:
%%writefile flash_attn.py
import torch
import triton
import triton.language as tl
import math

@triton.jit
def flash_attn_kernel(
    Q_ptr, K_ptr, V_ptr, O_ptr,
    stride_qb, stride_qh, stride_qm, stride_qk,
    stride_kb, stride_kh, stride_kn, stride_kk,
    stride_vb, stride_vh, stride_vn, stride_vk,
    stride_ob, stride_oh, stride_om, stride_ok,
    N, scale,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_D: tl.constexpr,
):
    batch_id = tl.program_id(0)
    head_id  = tl.program_id(1)
    block_id = tl.program_id(2)

    offs_m = block_id * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_d = tl.arange(0, BLOCK_D)

    Q_ptrs = (Q_ptr
              + batch_id * stride_qb
              + head_id  * stride_qh
              + offs_m[:, None] * stride_qm
              + offs_d[None, :] * stride_qk)

    q = tl.load(Q_ptrs, mask=offs_m[:, None] < N, other=0.0)

    m_i = tl.full([BLOCK_M], float('-inf'), dtype=tl.float32)
    l_i = tl.zeros([BLOCK_M], dtype=tl.float32)
    acc  = tl.zeros([BLOCK_M, BLOCK_D], dtype=tl.float32)

    for start_n in range(0, N, BLOCK_N):
        offs_n_cur = start_n + tl.arange(0, BLOCK_N)

        K_ptrs = (K_ptr
                  + batch_id * stride_kb
                  + head_id  * stride_kh
                  + offs_n_cur[:, None] * stride_kn
                  + offs_d[None, :] * stride_kk)
        k = tl.load(K_ptrs, mask=offs_n_cur[:, None] < N, other=0.0)

        scores = tl.dot(q, tl.trans(k)) * scale
        scores = tl.where(offs_n_cur[None, :] < N, scores, float('-inf'))

        m_new = tl.maximum(m_i, tl.max(scores, axis=1))
        alpha  = tl.exp(m_i - m_new)
        beta   = tl.exp(scores - m_new[:, None])
        l_i    = alpha * l_i + tl.sum(beta, axis=1)
        m_i    = m_new

        V_ptrs = (V_ptr
                  + batch_id * stride_vb
                  + head_id  * stride_vh
                  + offs_n_cur[:, None] * stride_vn
                  + offs_d[None, :] * stride_vk)
        v = tl.load(V_ptrs, mask=offs_n_cur[:, None] < N, other=0.0)
        acc = alpha[:, None] * acc + tl.dot(beta, v)

    acc = acc / l_i[:, None]

    O_ptrs = (O_ptr
              + batch_id * stride_ob
              + head_id  * stride_oh
              + offs_m[:, None] * stride_om
              + offs_d[None, :] * stride_ok)
    tl.store(O_ptrs, acc, mask=offs_m[:, None] < N)


def flash_attn_triton(Q, K, V):
    B, H, N, D = Q.shape
    O = torch.zeros_like(Q)

    BLOCK_M = 32
    BLOCK_N = 32
    BLOCK_D = triton.next_power_of_2(D)
    scale   = 1.0 / math.sqrt(D)   # Python float — safe for Triton

    grid = (B, H, triton.cdiv(N, BLOCK_M))

    flash_attn_kernel[grid](
        Q, K, V, O,
        Q.stride(0), Q.stride(1), Q.stride(2), Q.stride(3),
        K.stride(0), K.stride(1), K.stride(2), K.stride(3),
        V.stride(0), V.stride(1), V.stride(2), V.stride(3),
        O.stride(0), O.stride(1), O.stride(2), O.stride(3),
        N, scale,
        BLOCK_M=BLOCK_M,
        BLOCK_N=BLOCK_N,
        BLOCK_D=BLOCK_D,
    )
    return O


if __name__ == '__main__':
    device = 'cuda'
    B, H, N, D = 4, 8, 128, 64

    Q = torch.randn(B, H, N, D, device=device)
    K = torch.randn(B, H, N, D, device=device)
    V = torch.randn(B, H, N, D, device=device)

    for _ in range(10):
        out_triton = flash_attn_triton(Q, K, V)
    torch.cuda.synchronize()

    start = torch.cuda.Event(enable_timing=True)
    end   = torch.cuda.Event(enable_timing=True)

    start.record()
    for _ in range(100):
        out_triton = flash_attn_triton(Q, K, V)
    end.record()
    torch.cuda.synchronize()
    triton_ms = start.elapsed_time(end) / 100

    start.record()
    for _ in range(100):
        out_torch = torch.nn.functional.scaled_dot_product_attention(Q, K, V)
    end.record()
    torch.cuda.synchronize()
    torch_ms = start.elapsed_time(end) / 100

    max_diff = (out_triton - out_torch).abs().max().item()

    print(f"Triton flash-attn:      {triton_ms:.3f} ms")
    print(f"PyTorch SDPA (native):  {torch_ms:.3f} ms")
    print(f"Speedup:                {torch_ms/triton_ms:.2f}x")
    print(f"Max output difference:  {max_diff:.6f}")
    print(f"Correct (< 1e-2):       {max_diff < 1e-2}")

Overwriting flash_attn.py


In [13]:
!python flash_attn.py

Triton flash-attn:      0.333 ms
PyTorch SDPA (native):  0.571 ms
Speedup:                1.71x
Max output difference:  0.000000
Correct (< 1e-2):       True


"A Triton flash-attention kernel was implemented computing scaled dot-product attention online without materializing the full N×N score matrix, reducing peak memory from O(N²) to O(N). Benchmarked against PyTorch's native scaled_dot_product_attention on T4 (B=4, H=8, N=128, D=64), the Triton kernel achieved 0.333ms versus 0.571ms — a 1.71x speedup — while maintaining numerical exactness (max output difference: 0.000000). This demonstrates that memory hierarchy-aware kernel design, the same principle underlying flash attention in production LLM inference, can be implemented and validated in under 100 lines of Triton."